# Transfer learning: peptide single-task RT → lipid RT (Part A: pretrain + 5/25/50%)

Addresses Reviewer 1, comment 3: *transfer from a single-task peptide RT model to metabolites*. Provides the missing condition that disentangles the contributions of peptide pre-training (per se) from multi-task auxiliary supervision.

Mirrors `chemberta_TransferLearning_peptide_RTrdkit_Lipidmetlin_250k_A.ipynb` exactly except:

- The peptide source model is **ChemBERTa-RT (single-task, n_outputs=1)** instead of ChemBERTa+RDKit (multi-task, n_outputs=8). The peptide RT scaler is the only scaler shared with the lipid stage.

Pipeline:
1. Pretrain ChemBERTa-RT on the 250k peptide RT split.
2. Save encoder weights + RT scaler.
3. Transfer encoder to a lipid RT model (fresh single-output head). Scale lipid RT with the peptide scaler so the encoder's output range matches.
4. Fine-tune at 5%, 25%, 50% lipid subsets. Train from-scratch baselines alongside.
5. Save partial metrics for Part B to continue with 75%, 100%.

Run in Colab with a GPU runtime. Multi-seed support is in place but defaulted to `SEEDS = [0]` for the initial run.

In [ ]:
!pip install torch transformers scikit-learn matplotlib joblib

In [ ]:
# Upload: peptide250_train_with_rdkit.csv, peptide250_test_with_rdkit.csv,
#         METLIN_RT_lipid_train_with_rdkit.csv, METLIN_RT_lipid_test_with_rdkit.csv
n = 1
while n < 2:
    from google.colab import files
    uploaded = files.upload()
    n += 1

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error

tokenizer = AutoTokenizer.from_pretrained('seyonec/ChemBERTa-zinc-base-v1')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

peptide_train_df = pd.read_csv('/content/peptide250_train_with_rdkit.csv')
peptide_test_df  = pd.read_csv('/content/peptide250_test_with_rdkit.csv')
lipid_train_df   = pd.read_csv('/content/METLIN_RT_lipid_train_with_rdkit.csv')
lipid_test_df    = pd.read_csv('/content/METLIN_RT_lipid_test_with_rdkit.csv')
print(f'peptide  train: {len(peptide_train_df):>7d}   test: {len(peptide_test_df):>5d}')
print(f'lipid    train: {len(lipid_train_df):>7d}   test: {len(lipid_test_df):>5d}')

## 1. Source pretraining: ChemBERTa-RT (single-task) on peptides

In [ ]:
class SmilesRTDataset(Dataset):
    def __init__(self, smiles, rt_scaled, max_length=128):
        self.smiles = smiles
        self.rt = torch.tensor(rt_scaled, dtype=torch.float32)
        self.max_length = max_length
    def __len__(self):
        return len(self.smiles)
    def __getitem__(self, idx):
        enc = tokenizer(self.smiles[idx], padding='max_length', truncation=True,
                        max_length=self.max_length, return_tensors='pt')
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'rt':             self.rt[idx],
        }

class ChemBERTaRTRegressor(nn.Module):
    def __init__(self, n_outputs=1):
        super().__init__()
        self.bert = AutoModel.from_pretrained('seyonec/ChemBERTa-zinc-base-v1')
        hs = self.bert.config.hidden_size
        self.regressor = nn.Linear(hs, n_outputs)
    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return self.regressor(cls)

# Peptide RT scaler (peptide-only fit; this scaler is the bridge between peptide and lipid stages)
peptide_rt_scaler = MinMaxScaler().fit(peptide_train_df[['rt']].values)
peptide_rt_train  = peptide_rt_scaler.transform(peptide_train_df[['rt']].values).astype(np.float32).ravel()
peptide_rt_test   = peptide_rt_scaler.transform(peptide_test_df[['rt']].values).astype(np.float32).ravel()

peptide_train_ds = SmilesRTDataset(peptide_train_df['smile'].tolist(), peptide_rt_train)
peptide_test_ds  = SmilesRTDataset(peptide_test_df['smile'].tolist(),  peptide_rt_test)
peptide_train_loader = DataLoader(peptide_train_ds, batch_size=16, shuffle=True)
peptide_test_loader  = DataLoader(peptide_test_ds,  batch_size=16, shuffle=False)

EPOCHS = 15

torch.manual_seed(0); np.random.seed(0)
source_model = ChemBERTaRTRegressor(n_outputs=1).to(device)
optimizer = torch.optim.AdamW(source_model.parameters(), lr=2.5e-5)
criterion = nn.MSELoss()

def eval_singletask(model, loader, scaler):
    model.eval()
    preds, ys = [], []
    with torch.no_grad():
        for batch in loader:
            p = model(batch['input_ids'].to(device),
                      batch['attention_mask'].to(device)).squeeze(-1).cpu().numpy()
            preds.append(p); ys.append(batch['rt'].numpy())
    p = scaler.inverse_transform(np.concatenate(preds).reshape(-1, 1)).ravel()
    y = scaler.inverse_transform(np.concatenate(ys).reshape(-1, 1)).ravel()
    return r2_score(y, p), mean_absolute_error(y, p)

print('Pretraining ChemBERTa-RT on peptide RT...')
for epoch in range(1, EPOCHS + 1):
    source_model.train()
    for batch in peptide_train_loader:
        optimizer.zero_grad()
        pred = source_model(batch['input_ids'].to(device),
                            batch['attention_mask'].to(device)).squeeze(-1)
        loss = criterion(pred, batch['rt'].to(device))
        loss.backward()
        optimizer.step()
    r2_te, mae_te = eval_singletask(source_model, peptide_test_loader, peptide_rt_scaler)
    print(f'  epoch {epoch:2d}  test R2={r2_te:.4f}  MAE={mae_te:.2f}')

torch.save(source_model.bert.state_dict(), 'peptide_singletaskRT_bert_encoder.pth')
joblib.dump(peptide_rt_scaler, 'peptide_singletaskRT_rt_scaler.pkl')
print('saved peptide_singletaskRT_bert_encoder.pth, peptide_singletaskRT_rt_scaler.pkl')

## 2. Lipid stage: transfer encoder + fine-tune at 5%, 25%, 50%

For each fraction, we train two models on the same lipid subsample:

- **Transfer**: encoder initialized from the peptide-pretrained weights above, fresh single-output RT head.
- **Baseline**: vanilla ChemBERTa-zinc-base-v1, fresh single-output RT head. (Identical to the manuscript's existing from-scratch baseline.)

Lipid RT is scaled with the **peptide RT scaler** so the transferred encoder's output range matches what it was trained on.

In [ ]:
# Lipid RT scaling — share the peptide scaler, identical to the manuscript's existing transfer notebook
lipid_rt_train = peptide_rt_scaler.transform(lipid_train_df[['rt']].values).astype(np.float32).ravel()
lipid_rt_test  = peptide_rt_scaler.transform(lipid_test_df[['rt']].values).astype(np.float32).ravel()
lipid_train_smiles = lipid_train_df['smile'].tolist()
lipid_test_smiles  = lipid_test_df['smile'].tolist()

lipid_test_ds     = SmilesRTDataset(lipid_test_smiles, lipid_rt_test)
lipid_test_loader = DataLoader(lipid_test_ds, batch_size=16, shuffle=False)

PERCENTAGES_PART_A = [5, 25, 50]
SEEDS = [0]

def build_transfer_model(encoder_state_dict):
    m = ChemBERTaRTRegressor(n_outputs=1).to(device)
    m.bert.load_state_dict(encoder_state_dict)
    return m

def build_baseline_model():
    return ChemBERTaRTRegressor(n_outputs=1).to(device)

def train_lipid_model(model, train_loader, test_loader, epochs=EPOCHS):
    optimizer = torch.optim.AdamW(model.parameters(), lr=2.5e-5)
    criterion = nn.MSELoss()
    epoch_metrics = []
    for epoch in range(1, epochs + 1):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad()
            pred = model(batch['input_ids'].to(device),
                         batch['attention_mask'].to(device)).squeeze(-1)
            loss = criterion(pred, batch['rt'].to(device))
            loss.backward()
            optimizer.step()
        r2_tr, mae_tr = eval_singletask(model, train_loader, peptide_rt_scaler)
        r2_te, mae_te = eval_singletask(model, test_loader, peptide_rt_scaler)
        epoch_metrics.append({'epoch': epoch,
                              'r2_train': r2_tr, 'r2_test': r2_te,
                              'mae_train': mae_tr, 'mae_test': mae_te})
    return epoch_metrics

encoder_state = torch.load('peptide_singletaskRT_bert_encoder.pth', map_location=device)

all_rows = []
n_total = len(lipid_train_smiles)

for pct in PERCENTAGES_PART_A:
    n_sub = int(n_total * pct / 100)
    print(f'\n=== Lipid {pct}%  (n={n_sub})  ===')
    for seed in SEEDS:
        rng = np.random.RandomState(42 + seed)
        idx = rng.choice(n_total, size=n_sub, replace=False)
        sub_smiles = [lipid_train_smiles[i] for i in idx]
        sub_rt     = lipid_rt_train[idx]
        sub_ds     = SmilesRTDataset(sub_smiles, sub_rt)
        sub_loader = DataLoader(sub_ds, batch_size=16, shuffle=True)

        torch.manual_seed(seed); np.random.seed(seed)
        for variant_name, model_factory in [
            ('Transfer', lambda: build_transfer_model(encoder_state)),
            ('Baseline', build_baseline_model),
        ]:
            print(f'  seed={seed}  {variant_name}')
            model = model_factory()
            epoch_metrics = train_lipid_model(model, sub_loader, lipid_test_loader)
            for em in epoch_metrics:
                for split, r2_key, mae_key in [('Train', 'r2_train', 'mae_train'),
                                                ('Test', 'r2_test', 'mae_test')]:
                    all_rows.append({
                        'source_task': 'peptide-singletaskRT',
                        'percentage': pct, 'seed': seed,
                        'model': variant_name, 'epoch': em['epoch'],
                        'split': split,
                        'metric': 'R2',  'value': em[r2_key],
                    })
                    all_rows.append({
                        'source_task': 'peptide-singletaskRT',
                        'percentage': pct, 'seed': seed,
                        'model': variant_name, 'epoch': em['epoch'],
                        'split': split,
                        'metric': 'MAE', 'value': em[mae_key],
                    })
            del model
            torch.cuda.empty_cache() if device.type == 'cuda' else None

partial_df = pd.DataFrame(all_rows)
joblib.dump({'rows': all_rows, 'percentages_done': PERCENTAGES_PART_A, 'seeds': SEEDS},
            'lipid_metrics_partial_5_25_50_singletaskRT.pkl')
partial_df.to_csv('lipid_transfer_singletaskRT_partial.csv', index=False)
print('\nsaved lipid_metrics_partial_5_25_50_singletaskRT.pkl + .csv')

## 3. Hand-off to Part B

Download these three files for the next Colab session:

- `peptide_singletaskRT_bert_encoder.pth`
- `peptide_singletaskRT_rt_scaler.pkl`
- `lipid_metrics_partial_5_25_50_singletaskRT.pkl`

Then open `chemberta_TransferLearning_peptide-singletaskRT_to_Lipidmetlin_B.ipynb` for 75% and 100%.